In [1]:
from Model_MI_PyTorch import ef_net as ef_ef
from Model_WG_PyTorch import ef_net as ef_wg
from Metrics import train_metrics, save_seed
from Dataloader_EF import read_ef_train_si as si_data

import torch
from torch import nn
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score
import os


class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

		if config['mode'] == 0 or config['mode'] == 1:
			self.model = ef_ef().to(self.device)
		elif config['mode'] == 2:
			self.model = ef_wg().to(self.device)
		
		self.criterion = nn.CrossEntropyLoss()
		self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['learning_rate'])
		self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
			self.optimizer,
			mode='min',
			factor=0.8,
			patience=5,
			verbose=False,
			min_lr=0.1*config['learning_rate'],
		)

		self.results_dir = 'Results/' + config['exp_name']
		os.makedirs(self.results_dir, exist_ok=True)

	def z_score_mm(self, train_dataset, test_dataset, eps=1e-6):
		train_eeg = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
		train_eeg_mean = train_eeg.mean(dim=(0,1), keepdim=True)
		train_eeg_std = train_eeg.std(dim=(0,1),keepdim=True)
		train_eeg = (train_eeg - train_eeg_mean) / (train_eeg_std + eps)

		train_nirs = torch.stack([train_dataset[i][1] for i in range(len(train_dataset))])
		train_nirs_mean = train_nirs.mean(dim=(0,1), keepdim=True)
		train_nirs_std = train_nirs.std(dim=(0,1),keepdim=True)
		train_nirs = (train_nirs - train_nirs_mean) / (train_nirs_std + eps)

		train_labels = torch.stack([train_dataset[i][2] for i in range(len(train_dataset))])
		train_dataset = torch.utils.data.TensorDataset(train_eeg, train_nirs, train_labels)

		eval_eeg = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
		eval_eeg = (eval_eeg - train_eeg_mean) / (train_eeg_std + eps)
		eval_nirs = torch.stack([test_dataset[i][1] for i in range(len(test_dataset))])
		eval_nirs = (eval_nirs - train_nirs_mean) / (train_nirs_std + eps)
		eval_labels = torch.stack([test_dataset[i][2] for i in range(len(test_dataset))])
		test_dataset = torch.utils.data.TensorDataset(eval_eeg, eval_nirs, eval_labels)

		return train_dataset, test_dataset
	
	def train_subject(self, subject, mode):
		train_dataset, eval_dataset = si_data(subject, mode)
		if config['z_score']:
			train_dataset, eval_dataset = self.z_score_mm(train_dataset, eval_dataset)
		train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
	
		best_acc = 0
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		for epoch in range(self.config['num_epochs']):
			#----------Training----------
			self.model.train()
			total_correct, total_loss = 0, 0

			for batch_eeg, batch_nirs, batch_labels in train_loader:
				batch_eeg = batch_eeg.to(self.device, dtype=torch.float32)
				batch_nirs = batch_nirs.to(self.device, dtype=torch.float32)
				batch_labels = batch_labels.to(self.device)
				
				outputs = self.model(batch_eeg, batch_nirs)
				loss = self.criterion(outputs, batch_labels.long())
				self.optimizer.zero_grad()
				loss.backward()
				self.optimizer.step()
				
				preds = torch.argmax(outputs, dim=1)
				total_correct += (preds == batch_labels).sum().item()
				total_loss += loss.item()

			train_loss = total_loss / len(train_loader)
			train_acc = total_correct / len(train_loader.dataset)

			#----------Evaluation----------
			self.model.eval()
			total_loss, total_correct = 0, 0
			all_preds, all_labels = [], []
			
			with torch.no_grad():
				for eval_eeg, eval_nirs, eval_labels in eval_loader:
					eval_eeg = eval_eeg.to(self.device, dtype=torch.float32)
					eval_nirs = eval_nirs.to(self.device, dtype=torch.float32)
					eval_labels = eval_labels.to(self.device)
					
					outputs = self.model(eval_eeg, eval_nirs)
					loss = self.criterion(outputs, eval_labels.long())
					
					preds = torch.argmax(outputs, dim=1)
					total_correct += (preds == eval_labels).sum().item()
					total_loss += loss.item()
					
					all_preds.extend(preds.cpu().numpy())
					all_labels.extend(eval_labels.cpu().numpy())
			
			eval_loss = total_loss / len(eval_loader)
			self.scheduler.step(loss)
			eval_acc = total_correct / len(eval_loader.dataset)
			if eval_acc > best_acc:
				best_acc = eval_acc
				torch.save(self.model.state_dict(), self.results_dir + f'{subject}.pt')
			precision = precision_score(all_labels, all_preds, zero_division=0)
			recall = recall_score(all_labels, all_preds, zero_division=0)
			f1 = f1_score(all_labels, all_preds, zero_division=0)
			kappa = cohen_kappa_score(all_labels, all_preds)

			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject} | Epoch {epoch+1}/{self.config['num_epochs']} | LR {self.optimizer.param_groups[0]['lr']}")
			print(f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")

		train_metrics(self.results_dir, str(subject), acc_list, precision_list, recall_list, f1_list, kappa_list)

# Hyperparameters
config = {
	'num_classes': 2,
	'mi_root': '../../Dataset/EF-MI-MA/MI/',
	'ma_root': '../../Dataset/EF-MI-MA/MA/',
	'wg_root' : '../../Dataset/EF-WG/WG/',
	'batch_size': 64,
	'num_epochs': 120,
	'learning_rate': 1e-3,
	'mode': 0,
	'exp_name': '6020/',
	'z_score': True,
}

# Initialize and run trainer
for run in range(1):
	mode = config['mode']
	num_subject = (29 if mode == 0 or mode == 1 else 26) # MI = 0, MA = 1, WG = 2
	results_dir = 'Results/' + config['exp_name']
	seeds = []
	for subject in range(1, num_subject+1):
		seed = 42
		# seed = random.randint(1, 2025)
		seeds.append(seed)
		torch.manual_seed(seed)
		torch.cuda.manual_seed_all(seed)
		print(f"\n=== Subject {subject} ===")
		print(f'Seed is {seed}\n')
		trainer = Trainer(config)
		trainer.train_subject(subject, mode)
	
	save_seed(results_dir, mode, seeds)

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(



=== Subject 1 ===
Seed is 42



/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


1 Training Set EEG torch.Size([1680, 4000, 30, 1]) fNIRS torch.Size([1680, 200, 72, 1]) Labels torch.Size([1680])
1 Testing Set EEG torch.Size([60, 4000, 30, 1]) fNIRS torch.Size([60, 200, 72, 1]) Labels torch.Size([60])
Subject 1 | Epoch 1/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.50 | Eval Loss: 0.69 | Eval Acc: 0.50
Subject 1 | Epoch 2/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.59 | Eval Loss: 0.69 | Eval Acc: 0.52
Subject 1 | Epoch 3/120 | LR 0.001
Train Loss: 0.65 | Train Acc: 0.72 | Eval Loss: 0.73 | Eval Acc: 0.45
Subject 1 | Epoch 4/120 | LR 0.001
Train Loss: 0.60 | Train Acc: 0.73 | Eval Loss: 0.73 | Eval Acc: 0.53
Subject 1 | Epoch 5/120 | LR 0.001
Train Loss: 0.58 | Train Acc: 0.75 | Eval Loss: 0.78 | Eval Acc: 0.50
Subject 1 | Epoch 6/120 | LR 0.001
Train Loss: 0.56 | Train Acc: 0.74 | Eval Loss: 0.75 | Eval Acc: 0.55
Subject 1 | Epoch 7/120 | LR 0.0008
Train Loss: 0.55 | Train Acc: 0.75 | Eval Loss: 0.79 | Eval Acc: 0.48
Subject 1 | Epoch 8/120 | LR 0.0008
Train L

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


2 Training Set EEG torch.Size([1680, 4000, 30, 1]) fNIRS torch.Size([1680, 200, 72, 1]) Labels torch.Size([1680])
2 Testing Set EEG torch.Size([60, 4000, 30, 1]) fNIRS torch.Size([60, 200, 72, 1]) Labels torch.Size([60])
Subject 2 | Epoch 1/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.50 | Eval Loss: 0.69 | Eval Acc: 0.50
Subject 2 | Epoch 2/120 | LR 0.001
Train Loss: 0.69 | Train Acc: 0.58 | Eval Loss: 0.69 | Eval Acc: 0.58
Subject 2 | Epoch 3/120 | LR 0.001
Train Loss: 0.66 | Train Acc: 0.72 | Eval Loss: 0.71 | Eval Acc: 0.52
Subject 2 | Epoch 4/120 | LR 0.001
Train Loss: 0.60 | Train Acc: 0.74 | Eval Loss: 0.75 | Eval Acc: 0.52
Subject 2 | Epoch 5/120 | LR 0.001
Train Loss: 0.59 | Train Acc: 0.73 | Eval Loss: 0.79 | Eval Acc: 0.48
Subject 2 | Epoch 6/120 | LR 0.001
Train Loss: 0.56 | Train Acc: 0.75 | Eval Loss: 0.81 | Eval Acc: 0.47
Subject 2 | Epoch 7/120 | LR 0.001
Train Loss: 0.54 | Train Acc: 0.77 | Eval Loss: 0.74 | Eval Acc: 0.57
Subject 2 | Epoch 8/120 | LR 0.0008
Train Lo

KeyboardInterrupt: 